[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SergioPascualRedondo/Proyect_SP_AI_FOR_MEDICINE/blob/main/notebooks/heart_disease_clinical_stages.ipynb)

*Prof. Stefano Diciotti*  
*University of Bologna*  

---

**Academic Year:** 2025/2026  
**Student Name:** Sergio  
**Degree Program:** ____________________  
**Submission Date:** ____________________

# AI for Medicine Project - Clinically Staged Heart Disease Prediction

This project studies the Cleveland Heart Disease dataset as a binary classification problem. The target is the presence or absence of heart disease.

The main idea is not only to train a classifier, but to answer a clinically meaningful question:

**How much predictive performance is gained when progressively more specialised diagnostic information is added to routinely available cardiovascular risk factors?**

This follows the central principle of the AI for Medicine course: **the first step is to study the medical issue**, then design a reproducible and leakage-safe machine learning pipeline.

## Project Workflow

The notebook follows the same order as the project: understand the clinical problem, inspect the dataset, define clinically meaningful feature stages, build a leakage-safe pipeline, validate the models, and compare the final results by stage.


# 1. Clinical Problem and Dataset

Heart disease is a major clinical problem where early identification can support triage and prioritisation for further testing. In practice, not all information is available at the same time: routine risk factors are usually available earlier, while exercise-test and advanced diagnostic variables may require more time, resources, or specialist assessment.

For this reason, this project is not framed only as a binary classifier. The main question is whether useful prediction is possible before all diagnostic information is available.


## Clinical Motivation

The staged design compares how performance changes when progressively more clinical information is added. This allows the model to be interpreted as a possible support tool for early screening and diagnostic prioritisation, not as an automatic diagnosis system.


# 2. Initial Setup

The first cells prepare the notebook to run locally or in Google Colab, import the required libraries, and create the folders used to save figures and tables.


In [ ]:
# Path setup for local execution and Google Colab
import os
import sys
import subprocess
import importlib.util
import warnings
from pathlib import Path

REPO_URL = "https://github.com/SergioPascualRedondo/Proyect_SP_AI_FOR_MEDICINE.git"
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path("/content/Proyect_SP_AI_FOR_MEDICINE")
    if not PROJECT_ROOT.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(PROJECT_ROOT)])
    os.chdir(PROJECT_ROOT)
else:
    PROJECT_ROOT = Path.cwd().resolve()
    if PROJECT_ROOT.name == "notebooks":
        PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


In [ ]:
# Main dependencies
required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "shap": "shap",
}

missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report
from src.config import SEED, STAGE_FEATURES, TARGET
from src.data import load_heart_data, data_quality_report, split_features_target
from src.models import build_pipeline, candidate_models
from src.validation import make_locked_split, nested_cv_stage
from src.evaluation import evaluate_selected_strategy, evaluate_stages_on_final_test
from src.visualization import (
    save_figure,
    plot_target_distribution,
    plot_numeric_by_condition,
    plot_numeric_histograms,
    plot_categorical_proportions,
    plot_correlation_heatmap,
    plot_target_correlations,
    plot_stage_sizes,
    plot_cv_auc_by_stage,
    plot_cv_auc_lines,
    plot_threshold_tradeoff,
    save_threshold_analysis,
    stage_filename,
    plot_confusion_matrix_from_scores,
    plot_roc_curve_from_scores,
    plot_roc_curves_by_stage,
    save_shap_beeswarm,
    save_final_stage_plots,
)

np.random.seed(SEED)
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", message="Found unknown categories.*", category=UserWarning)


In [ ]:
# Output folders
FIG_EDA = PROJECT_ROOT / "results" / "figures" / "eda"
FIG_CV = PROJECT_ROOT / "results" / "figures" / "cv"
FIG_TEST = PROJECT_ROOT / "results" / "figures" / "final_test"
FIG_TEST_CONFUSION = FIG_TEST / "confusion_matrices_by_stage"
FIG_TEST_ROC = FIG_TEST / "roc_by_stage"
FIG_EXPLAIN = PROJECT_ROOT / "results" / "figures" / "explainability"

TAB_EDA = PROJECT_ROOT / "results" / "tables" / "eda"
TAB_CV = PROJECT_ROOT / "results" / "tables" / "cv"
TAB_TEST = PROJECT_ROOT / "results" / "tables" / "final_test"
TAB_EXPLAIN = PROJECT_ROOT / "results" / "tables" / "explainability"

for folder in [FIG_EDA, FIG_CV, FIG_TEST, FIG_TEST_CONFUSION, FIG_TEST_ROC, FIG_EXPLAIN, TAB_EDA, TAB_CV, TAB_TEST, TAB_EXPLAIN]:
    folder.mkdir(parents=True, exist_ok=True)


# 3. Data Loading, Quality Control and EDA

The project uses the cleaned Cleveland Heart Disease CSV. The target variable is binary: `condition = 0` means absence of heart disease, and `condition = 1` means presence of heart disease.


In [ ]:
df = load_heart_data()
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("Columns:")
print(df.columns.tolist())

## Variable Meaning

The dataset contains tabular clinical variables. Some are continuous measurements, some are binary variables, and others are categorical clinical findings that need to be encoded inside the pipeline.


## Data Quality Control

Before modelling, the dataset is checked for missing values, duplicated rows, unexpected question-mark values, class balance, and basic clinical plausibility.


In [ ]:
report = data_quality_report(df)

print("Rows:", report["n_rows"])
print("Columns:", report["n_columns"])
print("Duplicated rows:", report["duplicated_rows"])
print("\nMissing values per column:")
print(report["missing_values"])
print("\nQuestion mark values per column:")
print(report["question_mark_values"])
print("\nTarget counts:")
print(report["target_counts"])

In [ ]:
df.describe().T

## Data Quality Interpretation

The local file is already clean: no missing values, no duplicated rows, and no `?` values were found. Therefore, the main challenge is not heavy cleaning, but correct preprocessing, validation, and clinical interpretation.


## Exploratory Data Analysis

The following plots compare patients with and without heart disease and help identify which variables may be clinically informative before training any model.


In [ ]:
fig, ax = plot_target_distribution(df)
plt.show()

In [ ]:
print((df[TARGET].value_counts(normalize=True).sort_index() * 100).round(2))

The target is reasonably balanced. Therefore, the project can focus on clinical staging, validation, and threshold behaviour rather than class-imbalance corrections.

## Visual Summary


In [ ]:
fig, axes = plot_numeric_histograms(df, ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"])
save_figure(fig, FIG_EDA / "numeric_distributions_by_condition.png")
plt.show()

In [ ]:
fig, axes = plot_categorical_proportions(df, ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"])
save_figure(fig, FIG_EDA / "categorical_proportions_by_condition.png")
plt.show()

## Numeric Variables


In [ ]:
for column in ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]:
    fig, ax = plot_numeric_by_condition(df, column)
    plt.show()

## Categorical Variables


In [ ]:
categorical_to_plot = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

for column in categorical_to_plot:
    table = pd.crosstab(df[column], df[TARGET], normalize="columns") * 100
    print("\n", column)
    display(table.round(1))

## Correlation Check


In [ ]:
correlations = df.corr(numeric_only=True)[TARGET].drop(TARGET).sort_values(key=lambda s: s.abs(), ascending=False)
correlations.to_frame("correlation_with_condition")

In [ ]:
fig, ax = plot_correlation_heatmap(df)
save_figure(fig, FIG_EDA / "correlation_heatmap.png")
plt.show()

In [ ]:
fig, ax = plot_target_correlations(df)
save_figure(fig, FIG_EDA / "target_correlations.png")
plt.show()

# 4. Clinical Feature Stages

The variables are organised into four incremental clinical stages according to when they could realistically become available in a diagnostic pathway.


In [ ]:
for stage_name, features in STAGE_FEATURES.items():
    print(stage_name)
    print("Number of features:", len(features))
    print(features)
    print()

In [ ]:
fig, ax = plot_stage_sizes(STAGE_FEATURES)
save_figure(fig, FIG_EDA / "clinical_stage_sizes.png")
plt.show()

| Stage | Features added | Clinical interpretation |
|---|---|---|
| Stage 1 | `age`, `sex`, `trestbps`, `chol`, `fbs` | Routine cardiovascular risk factors |
| Stage 2 | + `cp`, `restecg` | Symptoms and resting ECG |
| Stage 3 | + `thalach`, `exang`, `oldpeak`, `slope` | Exercise stress test information |
| Stage 4 | + `ca`, `thal` | More advanced diagnostic measurements |

The comparison across stages is the central experiment of the project.

# 5. Pipeline and Leakage Control

The pipeline is the central technical element of the project. It keeps preprocessing and modelling together, so transformations are fitted only on training data during cross-validation and final testing.


## One Pipeline Template


In [ ]:
example_stage_name = "Stage 4 - Advanced diagnostic tests"
example_features = STAGE_FEATURES[example_stage_name]
example_model = candidate_models()["Logistic Regression"]["estimator"]

example_pipeline = build_pipeline(example_features, example_model)
example_pipeline


## Leakage Prevention and Audit

The pipeline protects mainly against L1 leakage, because imputing, scaling and one-hot encoding are fitted only inside the training folds.

The other leakage risks are handled by the study design: features are grouped by clinical availability to reduce L2 illegitimate-feature problems, and the split is performed at patient level to avoid non-independence between train and test samples.


# 6. Validation and Model Selection

The validation strategy uses a locked test set and nested cross-validation on the development set. This separates model selection from final evaluation.


In [ ]:
train_df, test_df = make_locked_split(df, test_size=0.2)

print("Development set:", train_df.shape)
print("Locked test set:", test_df.shape)
print("\nDevelopment target distribution:")
print(train_df[TARGET].value_counts(normalize=True).sort_index().round(3))
print("\nLocked test target distribution:")
print(test_df[TARGET].value_counts(normalize=True).sort_index().round(3))

## Nested Cross-Validation

The outer loop estimates generalisation performance, while the inner loop selects hyperparameters with `GridSearchCV`. ROC-AUC is used as the main selection metric because it evaluates discrimination across thresholds.


## Candidate Models

Logistic Regression is the primary model because it is simple and interpretable for a small tabular medical dataset. Random Forest is included as a secondary comparator for possible nonlinear relationships.


In [ ]:
models = candidate_models()

for model_name, model_spec in models.items():
    print(model_name)
    print(model_spec["param_grid"])
    print()

## Nested CV Experiment

The experiment compares both models across the four clinical stages using the same validation strategy.


In [ ]:
N_REPEATS = 5
all_results = []

for stage_name, stage_features in STAGE_FEATURES.items():
    for model_name, model_spec in models.items():
        print(f"Running: {stage_name} | {model_name}")
        stage_results = nested_cv_stage(
            train_df=train_df,
            stage_name=stage_name,
            stage_features=stage_features,
            model_name=model_name,
            model_spec=model_spec,
            n_repeats=N_REPEATS,
        )
        all_results.append(stage_results)

cv_results = pd.concat(all_results, ignore_index=True)
cv_results.head()

In [ ]:
cv_summary = (
    cv_results
    .groupby(["stage", "model"])[["roc_auc", "accuracy", "balanced_accuracy"]]
    .agg(["mean", "std"])
)
cv_summary.to_csv(TAB_CV / "nested_cv_summary.csv")
cv_summary

In [ ]:
fig, ax = plot_cv_auc_by_stage(cv_results)
save_figure(fig, FIG_CV / "nested_cv_auc_boxplot_by_stage.png")
plt.show()

In [ ]:
fig, ax = plot_cv_auc_lines(cv_results)
save_figure(fig, FIG_CV / "nested_cv_auc_line_by_stage.png")
plt.show()

## Cross-Validation Interpretation

The main result is the progressive improvement across clinical stages, especially when symptoms/resting ECG and later diagnostic variables are added.


## Final Strategy Selection

The final strategy is selected from development-set results before looking at the locked test set.


In [ ]:
mean_auc = (
    cv_results
    .groupby(["stage", "model"], as_index=False)["roc_auc"]
    .mean()
    .sort_values("roc_auc", ascending=False)
)
mean_auc

In [ ]:
best_row = mean_auc.iloc[0]
selected_stage = best_row["stage"]
selected_model = best_row["model"]
selected_features = STAGE_FEATURES[selected_stage]
selected_spec = models[selected_model]

print("Selected stage:", selected_stage)
print("Selected model:", selected_model)
print("Development ROC-AUC:", round(best_row["roc_auc"], 3))

# 7. Final Evaluation, Stage Comparison and Interpretation

After model selection, thresholds are inspected using development predictions and the locked test set is used for the final evaluation. The final comparison by stage directly answers the clinical question of the project.


In [ ]:
threshold_grid = np.round(np.arange(0.1, 1.0, 0.1), 2)
selected_eval = evaluate_selected_strategy(
    train_df, test_df, selected_features, selected_spec,
    build_pipeline, split_features_target, SEED,
    threshold_grid=threshold_grid,
    report_threshold=0.5,
)

fig, ax = save_threshold_analysis(
    selected_eval["dev_threshold_table"],
    table_path=TAB_CV / "development_threshold_metrics.csv",
    figure_path=FIG_CV / "development_threshold_tradeoff.png",
    title="Development threshold analysis",
)
plt.show()

selected_eval["dev_threshold_table"][["threshold", "sensitivity", "specificity", "precision", "accuracy", "balanced_accuracy"]]


## Final Test Evaluation


In [ ]:
fig, ax = save_threshold_analysis(
    selected_eval["test_threshold_table"],
    table_path=TAB_TEST / "final_test_threshold_metrics.csv",
    figure_path=FIG_TEST / "final_test_threshold_tradeoff.png",
    title="Final test threshold analysis",
)
plt.show()

selected_eval["test_threshold_table"][["threshold", "roc_auc", "sensitivity", "specificity", "precision", "accuracy", "balanced_accuracy", "tn", "fp", "fn", "tp"]]


In [ ]:
print(classification_report(
    selected_eval["y_test"],
    selected_eval["y_test_pred"],
    target_names=["No heart disease", "Heart disease"],
))


In [ ]:
fig, ax = plot_confusion_matrix_from_scores(
    selected_eval["y_test"],
    selected_eval["y_test_score"],
    selected_eval["report_threshold"],
    title="Final test confusion matrix",
)
save_figure(fig, FIG_TEST / "final_test_confusion_matrix.png")
plt.show()


In [ ]:
fig, ax = plot_roc_curve_from_scores(
    selected_eval["y_test"],
    selected_eval["y_test_score"],
    title="Final test ROC curve",
)
save_figure(fig, FIG_TEST / "final_test_roc_curve.png")
plt.show()


## Final Test Comparison by Clinical Stage

The same primary model is evaluated separately for each clinical stage. For each stage, the threshold is selected using only development predictions and then applied once to the locked test set.


In [ ]:
primary_model_name = "Logistic Regression"
primary_spec = models[primary_model_name]

stage_threshold_results, stage_test_results, stage_score_rows = evaluate_stages_on_final_test(
    train_df=train_df,
    test_df=test_df,
    stage_features=STAGE_FEATURES,
    model_name=primary_model_name,
    model_spec=primary_spec,
    build_pipeline=build_pipeline,
    split_features_target=split_features_target,
    seed=SEED,
    threshold_grid=threshold_grid,
)

stage_threshold_results.to_csv(TAB_CV / "development_threshold_metrics_by_stage.csv", index=False)
stage_test_results.to_csv(TAB_TEST / "final_test_metrics_by_stage.csv", index=False)

fig_cm, fig_roc = save_final_stage_plots(stage_score_rows, FIG_TEST_CONFUSION, FIG_TEST_ROC, FIG_TEST)
plt.show()

final_metric_columns = [
    "stage", "model", "selected_threshold_from_development", "roc_auc",
    "sensitivity", "specificity", "precision", "accuracy", "balanced_accuracy",
    "tn", "fp", "fn", "tp",
]
stage_test_results[final_metric_columns]


## SHAP Explanation of the Final Model

SHAP is used only as an interpretability tool for the final fitted model. It helps inspect which transformed clinical variables push predictions toward heart disease or no heart disease. This is not causal evidence.


In [ ]:
# Explain the final fitted Logistic Regression model.
shap_importance = save_shap_beeswarm(
    pipeline=selected_eval["search"].best_estimator_,
    X_background=selected_eval["X_dev"],
    X_to_explain=selected_eval["X_test"],
    figure_path=FIG_EXPLAIN / "shap_beeswarm_final_model.png",
    table_path=TAB_EXPLAIN / "shap_feature_importance.csv",
)

plt.show()
shap_importance.head(12)


## Saved Outputs

The notebook saves the main figures and tables in `results/`. The full clinical discussion, limitations, ethics and references are written in the final report.
